# Stage 2 neural comparison (Google Colab)

This thin runner clones the committed repository, installs its declared requirements, runs the configured Stage 2 search, and copies result CSV/JSON files to Google Drive. Select a GPU runtime in **Runtime → Change runtime type** before running all cells. The experiment itself is the package CLI; no model logic is duplicated in this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess
import tempfile
from pathlib import Path

repository_url = 'https://github.com/Nyamekye-tech/gru-mixed-adc-channel-estimator.git'
repository_path = Path(tempfile.mkdtemp(prefix='chest-stage2-'))
subprocess.run(
    ['git', 'clone', '--depth', '1', repository_url, str(repository_path)],
    check=True,
)
print(f'Cloned into {repository_path}')

In [ ]:
import sys
import torch

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'],
    cwd=repository_path,
    check=True,
)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Select a GPU runtime before running the neural search.')

In [ ]:
from datetime import datetime, timezone
import shutil
import sys

log_path = repository_path / 'stage2_run.log'
with log_path.open('w', encoding='utf-8') as log_file:
    run = subprocess.Popen(
        [
            sys.executable,
            '-u',
            '-m',
            'experiments.run_stage2',
            '--config',
            'configs/stage2.yaml',
        ],
        cwd=repository_path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if run.stdout is None:
        raise RuntimeError('Could not capture Stage 2 process output.')
    for line in run.stdout:
        print(line, end='', flush=True)
        log_file.write(line)
        log_file.flush()
    run.wait()
commit = subprocess.run(
    ['git', 'rev-parse', 'HEAD'],
    cwd=repository_path,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
drive_output = (
    Path('/content/drive/MyDrive/channel-estimator/stage2')
    / f'{commit[:12]}_{timestamp}'
)
drive_output.mkdir(parents=True, exist_ok=False)
result_paths = {
    'stage2_neural_results.csv': repository_path / 'results' / 'stage2_neural_results.csv',
    'stage2_neural_search.csv': repository_path / 'results' / 'stage2_neural_search.csv',
    'stage2_neural_metadata.json': repository_path / 'results' / 'stage2_neural_metadata.json',
    'stage2_run.log': log_path,
}
copied = []
for name, source in result_paths.items():
    if source.is_file():
        shutil.copy2(source, drive_output / name)
        copied.append(name)
print(f'Copied {copied} to {drive_output}')
if run.returncode != 0:
    raise RuntimeError(
        f'Stage 2 exited with status {run.returncode}; inspect '
        f'{drive_output / "stage2_run.log"} for the underlying error.'
    )